# பாடம் 16 - Microsoft Foundry உடன் அளவளாவிய முகவர்களை இயக்குதல்

இந்த நோட்புக் இல் நீங்கள் கற்பனைக்கான நிறுவனம் **Contoso**க்கான **உற்பத்தி தயாரான வாடிக்கையாளர் ஆதரவு முகவரை** உருவாக்குகிறீர்கள். முந்தைய பாடங்களின் போல அல்லாமல், இங்கு இயக்கத்தின் சிந்தனை சுற்றுப்பாதை முக்கியம் அல்ல — அது ஒரு முகவரை பாதுகாப்பாக அளவுகடந்த அளவில் இயக்கச் செய்யும் அதனை சுற்றியுள்ள அனைத்தும் ஆகும்:

1. **கருவி அழைக்கும் திறன்** — ஆர்டர் தேடல்கள் மற்றும் டிக்கெட் உருவாக்கல்.
2. **RAG** — அறிவுக்கூடத்திலிருந்து கொள்கை பதில்கள்.
3. **மனம்** — உரையாடல்களின் போது வாடிக்கையாளரை நினைவில் வைக்கும் தன்மை.
4. **மாதிரி வழித்தடம்** — எளிய கோரிக்கைகளை ஒரு சிறிய மாதிரிக்கு, சிக்கலானவைகளை பெரிய மாதிரிக்கு அனுப்புதல்.
5. **பதில் சேமிப்பு** — மாதிரி அழைப்பு இல்லாமல் திரும்பப் பேசப்படும் கேள்விகளை சேவையிட்டல்.
6. **மனித ஒப்புதல்பு** — ஒரு வரம்பை மீறும் பணப்பிரதி நிுத்தல் ஒப்புதல் பெறப்படுகிறது.
7. **மதிப்பீட்டுப் படுக்கை** — ஒரு கெட்ட வெளியீட்டை தடுக்கும் ஆஃப்லைன் சோதனை தொகுப்பு.
8. **பார்வையிடுதல்** — ஒவ்வொரு கோரிக்கைக்கும் OpenTelemetry கண்காணிப்பு.

ஒவ்வொரு பகுதியும் சுயமாக பயணிக்கக்கூடியதாகவும் இயக்கக்கூடியதாகவும் உள்ளது. ஒவ்வொரு வரியையும் படியுங்கள் — உற்பத்தி அடிப்படைகள் கவனமாக சிறியதாக வைத்துள்ளன.


## அமைவு

இந்த நோட்புக்கை இயக்கும் முன், நீங்கள் கீழ்கண்டவற்றை உறுதிப்படுத்திக்கொள்ள வேண்டும்:

1. **பரியிடப்பட்ட உரையாடல் மாடல் கொண்ட Microsoft Foundry திட்டம்** (உதா: `gpt-5-mini`).
2. **Azure CLI இல் உள்நுழைந்திருத்தல்** — உங்கள் டெர்மினல் இல் `az login` கட்டளை இயக்கவும்.
3. **தேவையான சூழல் மாறிலிகளை அமைத்தல்:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Microsoft Foundry திட்டத்தின் இறுதி புள்ளி.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் பரியிடப்பட்ட மாடலின் பெயர்.

RAG பகுதி `AZURE_SEARCH_SERVICE_ENDPOINT` மற்றும் `AZURE_SEARCH_API_KEY` அமைக்கப்பட்டால் **Azure AI Search** ஐ பயன்படுத்துகிறது, இல்லையெனில் நோட்புக் ஒரு தேடல் வளமின்றி இயக்க in-memory தேடலைFallback ஆகக் கொண்டுள்ளது.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. கருவிகள்

உற்பத்தி கருவிகள் உண்மையான அமைப்புகளுக்கு எதிராக உண்மையான பணிகளை செய்வவை. இங்கு நாம் ஒரு ஆர்டர் தரவுத்தளத்தை மற்றும் ஒரு டிக்கெட் அமைப்பை எளிய பைதான் செயல்பாடுகளுடன்ப்படும் முன்மாதிரியாக்குகிறோம். `@tool` அலங்காரி அவற்றை முகவரிக்கு வெளிப்படுத்துகிறது.

`issue_refund` அளவுக்குப்பெரிய பணத் திருப்பிகளுக்கு `approval_mode="always_require"` பயன்படுத்துகிறது என்பதை கவணியுங்கள் — இது மனிதர்போல் இயங்கும் அடிப்படை அம்சமாக நாம் பின்னர் பயன்படுத்துவோம்.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — கொள்கை அறிவுத்தளம்

கொள்கை செயல்பாட்டு கேள்விகள் ("உங்கள் திருப்பி கொடுக்கும் காலம் என்ன?") ஓர் அதிகாரப்பூர்வ மூலத்திலிருந்து பதிலளிக்கப்பட வேண்டும், மாதிரியின் நினைவகத்திலிருந்து அல்ல. நாங்கள் ஒரு சிறிய அறிவுத்தளத்தை தேடல் கருவியாக மடக்கிவைக்கிறோம்.

உற்பத்தியில் இது **Azure AI Search** ஆகும்; இங்கு கைநூல் எங்குமெல்லாம் இயங்கும் வகையில் நினைவகத்தில் அடிப்படையிலான முக்கியச் சொல் தேடலை வழங்குகிறோம், மேலும் சுற்றுச்சூழல் மாறிலிகள் கிடைக்கும்போது தானாகவே Azure AI Search க்கு மாறுகிறோம்.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. நினைவகம்

யாருடன் பேசுகிறோமோ அவரை மறந்துவிடும் ஆதரவு முகவர் ஒரு மோசமான ஆதரவு முகவர். நாங்கள் ஒவ்வொரு வாடிக்கையாளருக்குமான சிறிய சுயவிவரக் கடவுச்சொல் வைத்திருக்கிறோம் மற்றும் முகவரின் அறிவுரைகளில் ஒரு சிறிய சுருக்கத்தை சேர்க்கிறோம். உற்பத்தியில் இது ஒரு நினைவக சேவை ஆகும் (பாடம் 13 ஐ பாருங்கள்); இங்கே ஒரு dict இந்த மாதிரியை தெளிவாகக் காட்டுகிறது.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. மாதிரி வழிசெல் மற்றும் பதிலளிப்பு சேமிப்பு

இரண்டு செலவு மேலான்களை ஒரே கோரிக்கை கைமாற்றிலுள்ளே இணைத்துள்ளது:

- **வழிசெல்**: ஒரு மலிவு heuristical வகைப்பாட்டாளர் ஒரு கோரிக்கைக்கு சிறிய அல்லது பெரிய மாதிரி தேவைப்படுமா என தீர்மானிக்கின்றது.
- **சேமிப்பு**: ஒழுங்குபடுத்தப்பட்ட மீண்டும் கேள்விகள் எந்த மாதிரி அழைப்பும் இல்லாமல் நேரடியாக சேமிப்பிலிருந்து வழங்கப்படுகின்றன.

இங்கு வகைப்பாட்டாளர் விரும்புதலாக எளிமையானது. உற்பத்தியில் நீங்கள் அதை போக்குவரத்துடன் சரிபார்த்து, அதனை Foundry வின் மாதிரி வழிசெல்லுடன் மாற்றலாம்.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. முகவர், மனித அங்கீகாரம், மற்றும் கண்காணிப்பு

இப்போது நாம் மேலுள்ள கருவிகளிலிருந்து முகவரை உருவாக்கி ஒவ்வொரு கோரிக்கையையும் OpenTelemetry ஸ்பான் ஒன்றில் சுற்றி கொள்கிறோம். `handle_support_request` செயல்பாடு உற்பத்தி கோரிக்கை மேலாளர்: நினைவகக் காட்சி → வழிமாற்று → தொடர் → இயக்கு → நினைவகக் காட்சி.

மனித அங்கீகாரம் கட்டமைப்பால் கையாளப்படுகிறது: ஏனெனில் `issue_refund` என்பதில் `approval_mode="always_require"` உள்ளது, இயக்கம் இடைநிறுத்தப்படுகின்றது மற்றும் நாம் தெளிவாக தீர்மானிக்கும் ஒரு அங்கீகார கோரிக்கையை மேற்படுகிறது.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. கணக்கெடுப்பு வாயில்

பாடத்திலிருந்து இது வெளியீட்டு வாயில்: ஒரு ஆஃப்லைன் சோதனை தொகுப்பு முகவரியை மதிப்பீடு செய்கிறது, மற்றும் மேற்கொள்வது பாஸ் வீதம் ஒரு கருப்புள்ளியை கடந்தால் மட்டுமே நடைபெறும். இங்கு மதிப்பீடாளர் ஒரு எளிய விசைக் கருத்து சோதனை ஆகும், இது நோட்புக் சுயமாக்கப்பட்டிருக்க பயன்படுத்தப்படுகிறது; உற்பத்தியில் நீங்கள் ஒரு LLM-ஆக நீதியாளர் அல்லது ஒரு வடிவமைப்பாளர் மதிப்பீட்டியாளரை பயன்படுத்துவீர்கள் (பாடம் 10 ஐ காண்க).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ஒன்றிணைத்தல்: ஒரு மாதிரிப்படுத்தப்பட்ட வெளியீடு

கீழுள்ள செல்ஸ் முழு சுற்றையும் கற்பிப்பது போல காட்டுகிறது: மதிப்பீட்டு கதவை இயக்கவும், கடந்து சென்றால் மட்டுமே "வெளியிடவும்". இது நீங்கள் CI இல் இயக்கும் பாணி, ஒரு முகவர் பதிப்பை Foundry Agent Serviceக்கு உயர்த்துவதற்கு முன்.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## சுருக்கம்

நீங்கள் ஒவ்வொரு செயல்பாட்டு கவலைகளும் இணைக்கப்பட்டு தயாராக உள்ள வாடிக்கையாளர் ஆதரவு முகவரியை உருவாக்கியுள்ளீர்கள்:

- **கருவிகள், RAG மற்றும் நினைவகங்கள்** முகவரிக்கு திறனையும் சூழலைவும் வழங்குகின்றன.
- **மாதிரி வழிசெலுத்தல் மற்றும் சேமிப்பகம்** தாமதம் மற்றும் செலவை கட்டுப்படுத்துகின்றன.
- **மனித ஒப்புதல்** பெரும் பணமடுப்பு போன்ற பெரிய ஆபத்தான செயல்களை பாதுகாக்கிறது.
- **மதிப்பீட்டு கதவு** வெளியீடுகள் தவறாமல் முன் தடுக்கும்.
- **தொடர்தல்** ஒவ்வொரு கோரிக்கையையும் கண்காணிக்க உதவுகிறது.

### சவால்

இந்த முகவரியை விரிவடையச் செய்ய:

1. **பல மாதிரிகள் ஆதரவு** — மூன்று "கருத்தாக்கும்" அடுக்கு சேர்க்கவும் மற்றும் உயர்த்தல்களை/புகார்கள் அதற்கு வழிச்செலுத்தவும்.
2. **மதிப்பீட்டு கதவுகளைச் சேர்க்கவும்** — `TEST_CASES` ஐ பணமடுப்பு-ஒப்புதல் சூழல்களை கொண்டு விரிவாக்கி கதவு வழு மீள்வருகை பிடிப்பதை உறுதிப்படுத்தவும்.
3. **செலவு அறிந்த வழிசெலுத்தலைச் சேர்க்கவும்** — ஒரு கோரிக்கைக்கு மதிப்பிடப்பட்ட செலவை (சிறியது, பெரியது, சேமிப்பகம்) கண்காணித்து, கலந்த கேள்விகளின் தொகுப்புக்குப் பிறகு செலவு அறிக்கையை அச்சிடவும்.

அடுத்த பாடத்தில் நீங்கள் எதிர் பயணத்தை எடுத்துக் கொண்டு Microsoft Foundry Local மற்றும் Qwen உடன் உங்கள் இயந்திரத்தில் முழுமையாக ஒரு முகவரியை இயக்குவீர்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
